# GameHarbor — Limpieza de datos y Machine Learning con Árbol de Decisión
**Curso:** Soluciones Web y Aplicaciones Distribuidas — UPN
**Módulo:** Machine Learning y recomendaciones

**Objetivo del modelo:** predecir si a un usuario le **interesa** un videojuego del catálogo (lo compró, lo tiene en el carrito o lo vio ≥ 60 s) para generar recomendaciones personalizadas.

**Metodología (CRISP-DM):**
1. Comprensión de los datos → carga de la BD `DB_SW.txt` (script SQL Server de GameHarbor)
2. Preparación → diagnóstico y **limpieza de nulos / vacíos**
3. Ingeniería de variables → dataset usuario × juego
4. Modelado → `DecisionTreeClassifier` con búsqueda de hiperparámetros
5. Evaluación → métricas, matriz de confusión, árbol e importancia de variables
6. Despliegue → función de recomendación y modelo exportado (`.joblib`) para el backend

## 1. Librerías

In [ ]:
import re, os, glob, warnings
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score)
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED)

## 2. Carga de la base de datos desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUTA_BD = '/content/drive/MyDrive/Soluciones_Web/DB_SW/DB_SW.txt'
if not os.path.exists(RUTA_BD):                       # búsqueda alternativa
    encontrados = glob.glob('/content/drive/MyDrive/**/DB_SW.txt', recursive=True)
    if encontrados:
        RUTA_BD = encontrados[0]
    else:                                             # último recurso: subir el archivo
        from google.colab import files
        subido = files.upload()
        RUTA_BD = list(subido.keys())[0]

CARPETA_SALIDA = os.path.join(os.path.dirname(RUTA_BD), 'limpio')
os.makedirs(CARPETA_SALIDA, exist_ok=True)

with open(RUTA_BD, encoding='utf-8-sig') as f:
    sql = f.read()
print('Archivo:', RUTA_BD, '| caracteres:', len(sql))

## 3. Conversión del script SQL a tablas (DataFrames)
Se leen los `CREATE TABLE` (columnas, `IDENTITY`, `DEFAULT`) y los `INSERT INTO`, reproduciendo lo que haría SQL Server: las columnas no insertadas toman su valor por defecto o quedan en **NULL**.

In [ ]:
AHORA = pd.Timestamp(datetime.now()).floor('s')

def quitar_comentarios(texto):
    texto = re.sub(r'/\*.*?\*/', '', texto, flags=re.S)
    return re.sub(r'--[^\n]*', '', texto)

def dividir(texto, sep=','):
    # Divide por 'sep' respetando comillas y paréntesis
    partes, actual, nivel, en_cadena = [], '', 0, False
    i = 0
    while i < len(texto):
        c = texto[i]
        if c == "'":
            if en_cadena and i + 1 < len(texto) and texto[i+1] == "'":
                actual += "''"; i += 2; continue
            en_cadena = not en_cadena
        elif not en_cadena:
            if c == '(': nivel += 1
            elif c == ')': nivel -= 1
            elif c == sep and nivel == 0:
                partes.append(actual.strip()); actual = ''; i += 1; continue
        actual += c; i += 1
    if actual.strip(): partes.append(actual.strip())
    return partes

def valor_sql(tok):
    tok = tok.strip()
    if tok.upper() == 'NULL': return None
    m = re.fullmatch(r"N?'(.*)'", tok, flags=re.S)
    if m: return m.group(1).replace("''", "'")
    if re.fullmatch(r'SYSDATETIME\(\)|GETDATE\(\)', tok, flags=re.I): return AHORA
    m = re.fullmatch(r'DATEADD\(\s*(\w+)\s*,\s*(-?\d+)\s*,\s*(SYSDATETIME|GETDATE)\(\)\s*\)', tok, flags=re.I)
    if m:
        unidad = {'MINUTE': 'minutes', 'HOUR': 'hours', 'DAY': 'days', 'SECOND': 'seconds'}[m.group(1).upper()]
        return AHORA + timedelta(**{unidad: int(m.group(2))})
    if re.fullmatch(r'-?\d+', tok): return int(tok)
    if re.fullmatch(r'-?\d+\.\d+', tok): return float(tok)
    return tok

limpio = quitar_comentarios(sql)

# --- Estructura de las tablas
esquema = {}
for nombre, cuerpo in re.findall(r'CREATE\s+TABLE\s+(\w+)\s*\((.*?)\)\s*;', limpio, flags=re.S | re.I):
    cols = {}
    for linea in dividir(cuerpo):
        if re.match(r'\s*CONSTRAINT', linea, flags=re.I): continue
        partes = linea.split()
        if not partes: continue
        col = partes[0]
        identidad = bool(re.search(r'IDENTITY', linea, flags=re.I))
        m = re.search(r'DEFAULT\s*\((.*)\)\s*$', linea, flags=re.I)
        defecto = valor_sql(m.group(1)) if m else None
        cols[col] = {'identity': identidad, 'default': defecto}
    esquema[nombre.upper()] = cols

# --- Datos insertados
filas = {t: [] for t in esquema}
for tabla, columnas, valores in re.findall(r'INSERT\s+INTO\s+(\w+)\s*\(([^)]*)\)\s*VALUES\s*(.*?);', limpio, flags=re.S | re.I):
    columnas = [c.strip() for c in columnas.split(',')]
    for tupla in dividir(valores):
        tupla = tupla.strip()[1:-1]
        filas[tabla.upper()].append(dict(zip(columnas, [valor_sql(v) for v in dividir(tupla)])))

# --- Armado de DataFrames (IDENTITY 1..n, DEFAULT o NULL)
tablas = {}
for tabla, cols in esquema.items():
    registros = []
    for i, fila in enumerate(filas[tabla], start=1):
        reg = {}
        for col, info in cols.items():
            if col in fila: reg[col] = fila[col]
            elif info['identity']: reg[col] = i
            else: reg[col] = info['default']
        registros.append(reg)
    tablas[tabla] = pd.DataFrame(registros, columns=list(cols))

resumen = pd.DataFrame({t: {'filas': len(df), 'columnas': df.shape[1]} for t, df in tablas.items()}).T
resumen

In [ ]:
for t in ['USUARIO', 'JUEGO', 'SESION_JUEGO']:
    print(f'\n=== {t}')
    display(tablas[t])

## 4. Diagnóstico de valores nulos y vacíos

In [ ]:
def contar_vacios(df):
    # NULL + cadenas vacías o solo espacios
    vacios_txt = df.apply(lambda s: s.astype(str).str.strip().eq('') if s.dtype == object else pd.Series(False, index=s.index))
    return df.isna().sum() + vacios_txt.sum()

diag = []
for t, df in tablas.items():
    for col, n in contar_vacios(df).items():
        diag.append({'tabla': t, 'columna': col, 'nulos_o_vacios': int(n),
                     'porcentaje': round(100 * n / max(len(df), 1), 1)})
diag = pd.DataFrame(diag)
con_nulos = diag[diag.nulos_o_vacios > 0].sort_values('porcentaje', ascending=False)
print('Total de celdas nulas/vacías antes de limpiar:', int(diag.nulos_o_vacios.sum()))
con_nulos

In [ ]:
if len(con_nulos):
    plt.figure(figsize=(8, 0.5 * len(con_nulos) + 1.5))
    etiquetas = con_nulos.tabla + '.' + con_nulos.columna
    plt.barh(etiquetas, con_nulos.porcentaje, color='#c0504d')
    plt.xlabel('% de nulos / vacíos'); plt.title('Columnas con valores faltantes')
    plt.xlim(0, 100); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
else:
    print('No hay nulos ni vacíos.')

## 5. Limpieza
Reglas aplicadas a cada tabla:
1. Recortar espacios en textos y convertir cadenas vacías (`''`) en nulos.
2. **Eliminar columnas 100 % vacías** (no aportan información).
3. **Eliminar filas con nulos** en las columnas restantes.
4. Eliminar filas duplicadas.
5. Quitar datos sensibles que no se usan en el análisis (`USUARIO_PASSWORD_HASH`).
6. Ajustar tipos (fechas, números, booleanos).

In [ ]:
COLUMNAS_SENSIBLES = ['USUARIO_PASSWORD_HASH']
tablas_limpias, bitacora = {}, []

for t, df in tablas.items():
    d = df.copy()
    filas_ini, cols_ini = d.shape
    # 1) textos
    for c in d.select_dtypes(include='object'):
        d[c] = d[c].apply(lambda v: v.strip() if isinstance(v, str) else v).replace('', np.nan)
    # 2) columnas totalmente vacías
    cols_vacias = [c for c in d.columns if d[c].isna().all()]
    d = d.drop(columns=cols_vacias)
    # 5) sensibles
    d = d.drop(columns=[c for c in COLUMNAS_SENSIBLES if c in d.columns])
    # 3) filas con nulos
    d = d.dropna()
    # 4) duplicados (ignorando la columna ID)
    d = d.drop_duplicates(subset=list(d.columns[1:]) or None)
    # 6) tipos
    for c in d.columns:
        if 'FECHA' in c: d[c] = pd.to_datetime(d[c])
    for c in ['JUEGO_PRECIO', 'USUARIO_SALDO', 'ORDEN_TOTAL', 'ORDEN_DETALLE_PRECIO_UNITARIO',
              'SESION_BITRATE_MBPS', 'RECOMENDACION_PUNTAJE']:
        if c in d.columns: d[c] = d[c].astype(float)
    tablas_limpias[t] = d.reset_index(drop=True)
    bitacora.append({'tabla': t, 'filas_antes': filas_ini, 'filas_despues': len(d),
                     'columnas_antes': cols_ini, 'columnas_despues': d.shape[1],
                     'columnas_eliminadas': ', '.join(cols_vacias + [c for c in COLUMNAS_SENSIBLES if c in df.columns]) or '-'})

bitacora = pd.DataFrame(bitacora)
total_nulos = sum(int(contar_vacios(d).sum()) for d in tablas_limpias.values())
print('Celdas nulas/vacías después de limpiar:', total_nulos)
bitacora

In [ ]:
# Exportar tablas limpias a Drive (CSV por tabla + un Excel con todas)
ruta_excel = os.path.join(CARPETA_SALIDA, 'GameHarbor_limpio.xlsx')
with pd.ExcelWriter(ruta_excel) as xw:
    for t, d in tablas_limpias.items():
        d.to_csv(os.path.join(CARPETA_SALIDA, f'{t}.csv'), index=False, encoding='utf-8-sig')
        d.to_excel(xw, sheet_name=t[:31], index=False)
    bitacora.to_excel(xw, sheet_name='BITACORA_LIMPIEZA', index=False)
print('Tablas limpias guardadas en:', CARPETA_SALIDA)

## 6. Ingeniería de variables: dataset usuario × juego
Una fila por cada combinación **cliente – juego**.

| Variable | Descripción |
|---|---|
| `JUEGO_PRECIO` | Precio del juego |
| `JUEGO_REQUIERE_CLOUD` | 1 si se juega en la nube |
| `JUEGO_DISPONIBLE` | 1 si su estado es DISPONIBLE |
| `ANTIGUEDAD_DIAS` | Días desde el lanzamiento |
| `USUARIO_SALDO` | Saldo del usuario |
| `PUEDE_PAGAR` | 1 si saldo ≥ precio |
| `AFINIDAD_CATEGORIA` | 1 si el juego comparte categoría con los juegos que el usuario reseñó |
| `CAT_*` | Categorías del juego (one-hot) |

**Variable objetivo `INTERES`** = 1 si el usuario tiene el juego en su biblioteca, en su carrito o lo vio ≥ 60 segundos.

In [ ]:
T = tablas_limpias
cat = T['JUEGO_CATEGORIA'].merge(T['CATEGORIA'], on='CATEGORIA_ID')
cat_onehot = pd.crosstab(cat.JUEGO_ID, cat.CATEGORIA_NOMBRE).clip(upper=1).add_prefix('CAT_').reset_index()
cats_por_juego = cat.groupby('JUEGO_ID').CATEGORIA_ID.apply(set).to_dict()

juegos = T['JUEGO'].merge(cat_onehot, on='JUEGO_ID', how='left').fillna(0)
juegos['JUEGO_DISPONIBLE'] = (juegos.JUEGO_ESTADO == 'DISPONIBLE').astype(int)
juegos['ANTIGUEDAD_DIAS'] = (AHORA - juegos.JUEGO_FECHA_LANZAMIENTO).dt.days

clientes_ids = T['USUARIO_ROL'].merge(T['ROL'], on='ROL_ID').query("ROL_NOMBRE == 'CLIENTE'").USUARIO_ID
clientes = T['USUARIO'][T['USUARIO'].USUARIO_ID.isin(clientes_ids)]

# Preferencias del usuario = categorías de los juegos que reseñó
pref = (T['RESENA'].assign(cats=T['RESENA'].JUEGO_ID.map(cats_por_juego))
        .groupby('USUARIO_ID').cats.apply(lambda s: set().union(*s)).to_dict())

# Señales de interés (objetivo)
biblioteca = set(zip(T['BIBLIOTECA'].USUARIO_ID, T['BIBLIOTECA'].JUEGO_ID))
carrito = T['CARRITO_ITEM'].merge(T['CARRITO'], on='CARRITO_ID')
carrito = set(zip(carrito.USUARIO_ID, carrito.JUEGO_ID))
vistas = T['HISTORIAL_VISTA'].query('HISTORIAL_SEGUNDOS >= 60')
vistas = set(zip(vistas.USUARIO_ID, vistas.JUEGO_ID))

COLS_CAT = [c for c in juegos.columns if c.startswith('CAT_')]
FEATURES = ['JUEGO_PRECIO', 'JUEGO_REQUIERE_CLOUD', 'JUEGO_DISPONIBLE', 'ANTIGUEDAD_DIAS',
            'USUARIO_SALDO', 'PUEDE_PAGAR', 'AFINIDAD_CATEGORIA'] + COLS_CAT

reales = clientes[['USUARIO_ID', 'USUARIO_NOMBRE', 'USUARIO_SALDO']].merge(juegos, how='cross')
reales['PUEDE_PAGAR'] = (reales.USUARIO_SALDO >= reales.JUEGO_PRECIO).astype(int)
reales['AFINIDAD_CATEGORIA'] = [int(bool(pref.get(u, set()) & cats_por_juego.get(j, set())))
                                for u, j in zip(reales.USUARIO_ID, reales.JUEGO_ID)]
reales['INTERES'] = [int((u, j) in biblioteca | carrito | vistas) for u, j in zip(reales.USUARIO_ID, reales.JUEGO_ID)]
reales[FEATURES] = reales[FEATURES].astype(float)

print(f'Dataset real: {len(reales)} filas ({clientes.shape[0]} clientes × {juegos.shape[0]} juegos)')
reales[['USUARIO_NOMBRE', 'JUEGO_TITULO'] + FEATURES + ['INTERES']]

## 7. Ampliación del dataset (datos **simulados**)
La BD de ejemplo solo tiene 3 clientes → 15 filas, insuficiente para entrenar y evaluar un modelo de forma confiable.
Para la demostración se generan **usuarios simulados** sobre el **catálogo real**, con reglas de comportamiento plausibles
(más interés si hay afinidad de categoría, si puede pagar y si el juego está disponible). Los datos reales se reservan
para validar el modelo al final. *En producción, este paso se reemplaza por el historial real de la plataforma.*

In [ ]:
rng = np.random.default_rng(SEED)
N_USUARIOS_SIM = 400
todas_cats = sorted(set(T['CATEGORIA'].CATEGORIA_ID))

sim = []
for u in range(N_USUARIOS_SIM):
    saldo = float(rng.choice([0, 20, 50, 80, 100, 150]))
    prefs = set(rng.choice(todas_cats, size=rng.integers(1, 3), replace=False))
    for _, j in juegos.iterrows():
        afin = int(bool(prefs & cats_por_juego.get(j.JUEGO_ID, set())))
        puede = int(saldo >= j.JUEGO_PRECIO)
        logit = (-1.6 + 2.4 * afin + 0.9 * puede + 0.8 * j.JUEGO_DISPONIBLE
                 - 0.03 * (j.JUEGO_PRECIO - 40) - 0.0006 * (j.ANTIGUEDAD_DIAS - 500))
        interes = int(rng.random() < 1 / (1 + np.exp(-logit)))
        fila = {c: j[c] for c in ['JUEGO_PRECIO', 'JUEGO_REQUIERE_CLOUD', 'JUEGO_DISPONIBLE', 'ANTIGUEDAD_DIAS'] + COLS_CAT}
        fila.update(USUARIO_SALDO=saldo, PUEDE_PAGAR=puede, AFINIDAD_CATEGORIA=afin, INTERES=interes)
        sim.append(fila)
sim = pd.DataFrame(sim)[FEATURES + ['INTERES']].astype(float)
sim['INTERES'] = sim.INTERES.astype(int)

print('Filas simuladas:', len(sim), '| nulos:', int(sim.isna().sum().sum()))
print(sim.INTERES.value_counts(normalize=True).rename({0: 'Sin interés', 1: 'Con interés'}).round(3))
sim.describe().T[['mean', 'min', 'max']]

## 8. División entrenamiento / prueba (80 / 20, estratificada)

In [ ]:
X, y = sim[FEATURES], sim['INTERES']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

## 9. Entrenamiento del Árbol de Decisión
Se buscan los mejores hiperparámetros con validación cruzada de 5 particiones (`GridSearchCV`), optimizando F1.
Limitar la profundidad evita el sobreajuste y mantiene el árbol interpretable.

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4],          # árbol interpretable para el informe
    'min_samples_leaf': [5, 10, 20],
}
grid = GridSearchCV(DecisionTreeClassifier(random_state=SEED, class_weight='balanced'),
                    param_grid, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                    scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)
modelo = grid.best_estimator_
print('Mejores hiperparámetros:', grid.best_params_)
print('F1 promedio en validación cruzada:', round(grid.best_score_, 3))

## 10. Evaluación en el conjunto de prueba

In [ ]:
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]
metricas = pd.Series({
    'Exactitud (accuracy)': accuracy_score(y_test, y_pred),
    'Precisión': precision_score(y_test, y_pred),
    'Sensibilidad (recall)': recall_score(y_test, y_pred),
    'F1-score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
}).round(3)
display(metricas.to_frame('valor'))
print(classification_report(y_test, y_pred, target_names=['Sin interés', 'Con interés']))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['Sin interés', 'Con interés']).plot(cmap='Blues')
plt.title('Matriz de confusión — Árbol de decisión'); plt.grid(False); plt.show()

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(modelo, feature_names=FEATURES, class_names=['Sin interés', 'Con interés'],
          filled=True, rounded=True, fontsize=9, impurity=False, proportion=True)
plt.title('Árbol de decisión — Interés del usuario en un videojuego', fontsize=14)
plt.show()

print(export_text(modelo, feature_names=FEATURES))

In [ ]:
imp = pd.Series(modelo.feature_importances_, index=FEATURES).sort_values()
imp = imp[imp > 0]
plt.figure(figsize=(7, 0.45 * len(imp) + 1.5))
plt.barh(imp.index, imp.values, color='#4f81bd')
plt.xlabel('Importancia'); plt.title('Importancia de las variables'); plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).round(3).to_frame('importancia')

## 11. Validación con los datos reales de GameHarbor

In [ ]:
reales['PROB_INTERES'] = modelo.predict_proba(reales[FEATURES])[:, 1].round(3)
reales['PREDICCION'] = modelo.predict(reales[FEATURES])
print('Nota: solo 15 filas reales; esta cifra es referencial.')
print('Exactitud sobre datos reales:', round(accuracy_score(reales.INTERES, reales.PREDICCION), 3),
      f'({(reales.INTERES == reales.PREDICCION).sum()}/{len(reales)} aciertos)')
reales[['USUARIO_NOMBRE', 'JUEGO_TITULO', 'AFINIDAD_CATEGORIA', 'PUEDE_PAGAR', 'INTERES', 'PREDICCION', 'PROB_INTERES']]

## 12. Recomendaciones personalizadas
Para cada cliente se ordenan los juegos **disponibles** que **aún no tiene en su biblioteca** según la probabilidad de interés.
El resultado tiene el mismo formato que la tabla `RECOMENDACION` (`USUARIO_ID`, `JUEGO_ID`, `RECOMENDACION_PUNTAJE`, `RECOMENDACION_ORIGEN = 'ML'`).

In [ ]:
def recomendar(usuario_id, top_n=3):
    cand = reales[(reales.USUARIO_ID == usuario_id) & (reales.JUEGO_DISPONIBLE == 1) &
                  (~reales.apply(lambda r: (r.USUARIO_ID, r.JUEGO_ID) in biblioteca, axis=1))]
    return (cand.sort_values('PROB_INTERES', ascending=False).head(top_n)
            [['USUARIO_ID', 'USUARIO_NOMBRE', 'JUEGO_ID', 'JUEGO_TITULO', 'PROB_INTERES']])

recs = pd.concat([recomendar(u) for u in clientes.USUARIO_ID], ignore_index=True)
display(recs)

tabla_recomendacion = recs.rename(columns={'PROB_INTERES': 'RECOMENDACION_PUNTAJE'})[['USUARIO_ID', 'JUEGO_ID', 'RECOMENDACION_PUNTAJE']]
tabla_recomendacion['RECOMENDACION_ORIGEN'] = 'ML'
tabla_recomendacion.to_csv(os.path.join(CARPETA_SALIDA, 'RECOMENDACION_ML.csv'), index=False)

# Script SQL listo para cargar las recomendaciones en SQL Server
inserts = ',\n'.join(f"({r.USUARIO_ID}, {r.JUEGO_ID}, {r.RECOMENDACION_PUNTAJE:.4f}, 'ML')" for r in tabla_recomendacion.itertuples())
sql_rec = ('DELETE FROM RECOMENDACION WHERE RECOMENDACION_ORIGEN = \'ML\';\n'
           'INSERT INTO RECOMENDACION (USUARIO_ID, JUEGO_ID, RECOMENDACION_PUNTAJE, RECOMENDACION_ORIGEN) VALUES\n'
           + inserts + ';')
print(sql_rec)

## 13. Exportar el modelo para integrarlo con el backend

In [ ]:
ruta_modelo = os.path.join(CARPETA_SALIDA, 'modelo_arbol_gameharbor.joblib')
joblib.dump({'modelo': modelo, 'features': FEATURES}, ruta_modelo)
print('Modelo guardado en:', ruta_modelo)

# Ejemplo de uso desde un servicio Python (API Flask/FastAPI consumida por el backend Java):
cargado = joblib.load(ruta_modelo)
ejemplo = pd.DataFrame([dict.fromkeys(cargado['features'], 0.0) | {
    'JUEGO_PRECIO': 39.9, 'JUEGO_REQUIERE_CLOUD': 1, 'JUEGO_DISPONIBLE': 1, 'ANTIGUEDAD_DIAS': 500,
    'USUARIO_SALDO': 50, 'PUEDE_PAGAR': 1, 'AFINIDAD_CATEGORIA': 1, 'CAT_RPG': 1}])
print('Probabilidad de interés del ejemplo:', round(cargado['modelo'].predict_proba(ejemplo[cargado['features']])[0, 1], 3))

## 14. Conclusiones
- La limpieza eliminó las columnas totalmente vacías (p. ej. `JUEGO_IMAGEN_URL`), los datos sensibles no necesarios y cualquier fila con nulos, dejando **0 celdas vacías** en las tablas exportadas (`limpio/`).
- El **árbol de decisión** es interpretable: sus reglas (ver `export_text`) muestran que la **afinidad de categoría** y la **capacidad de pago** son los factores que más influyen en el interés.
- Las métricas de la sección 10 se obtuvieron con datos simulados sobre el catálogo real; al acumular historial real (vistas, carritos, compras) en Azure SQL, basta con reemplazar la sección 7 por esos datos y reentrenar.
- El modelo exportado (`.joblib`) y el script `INSERT` permiten poblar la tabla `RECOMENDACION` del sistema GameHarbor.